# Experiment 3 — MovieLens random pairs: robustness across the real-data pair distribution

Figures per `docs/experiments.md`, Experiment 3 "Figures" subsection. This notebook only loads `results/exp3_1/results.parquet`, `results/exp3_2/results.parquet`, `results/exp3_3/results.parquet` and plots — it never recomputes (see CLAUDE.md and `docs/experiments.md` §0.4). Run `python -m scripts.run_experiment3` (or `SMOKE_TEST=1 python -m scripts.run_experiment3` first) to generate them.

**Note:** this experiment was originally built and run as "Experiment 4"; a separate, older, fixed-instance "Experiment 3" has been dropped entirely and this took its number (see `docs/experiments.md`'s Experiment 3 "RENUMBERING NOTE" and "REVISION 4"). REVISION 4 also changed the algorithm roster, version thresholds, `budget_cap`, and scale — this notebook has not yet been run against real results under this new config (only the logic was smoke-tested); re-run once `results/exp3_*/results.parquet` reflect a real run.

Each version reports 6 algorithm rows per rep: `kabc`, `kabc-0.5x` (kabc re-run at 0.5× its own realized τ — tests how conservative its stopping rule is), `acb`, `tas-fw`, and two `unif-kkm` variants (`unif-kkm-1x`/`unif-kkm-0.5x` = that same rep's kabc τ, ×1 or ×0.5, capped at `budget_cap` — see docs/experiments.md's "KABC-0.5X and UNIF-KKM VARIANTS"). **Multiplier note (2026-09-14):** this was originally 0.9x; changed to 0.5x once direct per-rep inspection (Experiment 2's data) showed KABC's doubling-trick schedule quantizes its own realized τ to a small set of round-boundary checkpoints (the final round alone typically costs roughly half of the total realized τ), so a 0.9x cut cannot actually test "10% less budget" — it almost always lands in the same "one checkpoint short" bucket as a much larger cut would.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kabc.plotting import set_style, savefig, ALGO_ORDER, ALGO_COLORS, ALGO_LABELS

set_style()

VERSIONS = ["exp3_1", "exp3_2", "exp3_3"]
VERSION_TITLES = {
    "exp3_1": "exp3_1: tight gap, argmax MMD² (gap ≤ 0.25)",
    "exp3_2": "exp3_2: moderate-large gap, uniform (0.25 ≤ gap ≤ 1.0)",
    "exp3_3": "exp3_3: tight gap, uniform (gap ≤ 0.25)",
}

dfs = {}
for version in VERSIONS:
    path = Path(f"../results/{version}/results.parquet")
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run `python -m scripts.run_experiment3` (or with SMOKE_TEST=1) "
            "to generate it -- this notebook only plots existing results, it never recomputes them."
        )
    dfs[version] = pd.read_parquet(path)
    print(f"loaded {len(dfs[version])} rows from {path}")

dfs[VERSIONS[0]].head()


## Figures: four bar plots (ARI, τ, wall-clock/rep, convergence rate), each with all 3 versions × 6 algorithms

One bar plot per metric, not one figure per version: each plot has 18 bars grouped into 3 sets of 6 (one set per version: `exp3_1`, `exp3_2`, `exp3_3`), each set showing all 6 algorithm rows (`kabc`, `kabc-0.5x`, `acb`, `tas-fw`, `unif-kkm-1x`, `unif-kkm-0.5x`) side by side. Bar height = mean over reps; error bar = standard deviation over reps (not standard error — the spread itself is the point, since each rep is also a different randomly-sampled movie pair, not just different pull-noise on a fixed instance).

The convergence-rate plot (new, REVISION 4) is the fraction of reps with `stopped=True` per algorithm per version, using the same mean/std-over-reps convention (a Bernoulli mean ± its std across reps). Read it alongside the ARI plot, not in isolation: per `docs/experiments.md`'s caution, kabc's graceful trivial-cluster fallback (ARI exactly 0.0, `stopped=False`) can make a forced-guess baseline's ARI look artificially competitive, and a lower ARI for `kabc-0.5x` vs. primary `kabc` is the INTENDED signal (how much the extra half of budget bought), not a failure.


In [ ]:
ALGOS = [a for a in ALGO_ORDER if a in dfs[VERSIONS[0]]["algorithm"].unique()]
assert len(ALGOS) == 6, f"expected 6 algorithm rows, got {len(ALGOS)}: {ALGOS}"

VERSION_SHORT = {"exp3_1": "3_1", "exp3_2": "3_2", "exp3_3": "3_3"}


def grouped_bar_plot(metric_col: str, ylabel: str, fig_name: str, log_scale: bool = False):
    """One bar plot: 3 sets (versions) x 6 bars (algorithms) = 18 bars total,
    grouped with a visible gap between sets so the 3 versions read as
    distinct clusters, not one continuous row of 18 bars."""
    n_algo = len(ALGOS)
    bar_width = 0.8 / n_algo
    group_gap = 0.6  # extra spacing between each version's cluster of 6 bars

    fig, ax = plt.subplots(figsize=(12, 4))
    xtick_positions, xtick_labels = [], []
    for v_idx, version in enumerate(VERSIONS):
        df = dfs[version]
        group_center = v_idx * (1.0 + group_gap)
        for a_idx, algo in enumerate(ALGOS):
            sub = df[df["algorithm"] == algo]
            mean = sub[metric_col].mean()
            std = sub[metric_col].std()
            x = group_center + (a_idx - (n_algo - 1) / 2) * bar_width
            ax.bar(x, mean, yerr=std, width=bar_width * 0.92, capsize=2,
                   color=ALGO_COLORS[algo], alpha=0.88,
                   label=ALGO_LABELS[algo] if v_idx == 0 else None)
        xtick_positions.append(group_center)
        xtick_labels.append(VERSION_SHORT[version])

    ax.set_xticks(xtick_positions)
    ax.set_xticklabels(xtick_labels, fontsize=9)
    ax.set_xlabel("version")
    ax.set_ylabel(ylabel)
    if log_scale:
        ax.set_yscale("log")
    ax.legend(fontsize=6.5, frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.15))
    fig.tight_layout()
    savefig(fig, fig_name, fig_dir="../results/figures")
    plt.show()


grouped_bar_plot("ari", "mean ARI (± std)", "exp3_fig1_ari_bars")
grouped_bar_plot("tau", r"mean $\tau$ (± std)", "exp3_fig2_tau_bars", log_scale=True)
grouped_bar_plot("wall_clock_seconds", "mean wall-clock/rep, s (± std)", "exp3_fig3_wallclock_bars", log_scale=True)
grouped_bar_plot("stopped", "mean convergence rate (± std)", "exp3_fig4_convergence_bars")


## Summary tables: exact mean ± std per algorithm, per version

Printed (not just plotted) so the restricted-budget-style claims in the write-up can cite exact numbers, same convention as notebook 01's restricted-budget breakdowns. `frac_stopped` here is the same convergence-rate quantity as the new Figure 4 bar plot above, in table form.


In [ ]:
for version in VERSIONS:
    df = dfs[version]
    algos_present = [a for a in ALGO_ORDER if a in df["algorithm"].unique()]
    rows = []
    for a in algos_present:
        sub = df[df["algorithm"] == a]
        rows.append({
            "algorithm": ALGO_LABELS[a],
            "mean_tau": sub["tau"].mean(),
            "std_tau": sub["tau"].std(),
            "mean_ari": sub["ari"].mean(),
            "std_ari": sub["ari"].std(),
            "mean_wall_clock_s": sub["wall_clock_seconds"].mean(),
            "std_wall_clock_s": sub["wall_clock_seconds"].std(),
            "frac_stopped": sub["stopped"].mean(),
            "std_stopped": sub["stopped"].std(),
        })
    print(f"=== {version} ===")
    display(pd.DataFrame(rows))
